# Laboratorio: transformaciones lineales y matrices

En este laboratorio trabajaremos con la misma transformación en varias representaciones. Al finalizar podrás:

- construir una matriz a partir de las imágenes de una base;
- calcular núcleo, imagen, rango y nulidad;
- verificar inyectividad y sobreyectividad;
- obtener matrices relativas a bases distintas;
- comprobar composición e inversa mediante matrices.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=5, suppress=True)

## 1. La matriz se construye con imágenes de una base

Consideremos $T:\mathbb R^3\to\mathbb R^2$ dada por $T(x,y,z)=(x+y,y+z)$. Las columnas de su matriz canónica son $T(e_1)$, $T(e_2)$ y $T(e_3)$.

In [ ]:
def T(v):
    x, y, z = sp.Matrix(v)
    return sp.Matrix([x + y, y + z])

E3 = [sp.eye(3).col(j) for j in range(3)]
A = sp.Matrix.hstack(*(T(e) for e in E3))
print("Matriz canónica de T:")
sp.pprint(A)
assert A == sp.Matrix([[1, 1, 0], [0, 1, 1]])

In [ ]:
v = sp.Matrix([2, -1, 4])
print("T(v) mediante la regla:", list(T(v)))
print("T(v) mediante la matriz:", list(A * v))
assert T(v) == A * v

## 2. Núcleo, imagen y rango-nulidad

SymPy devuelve bases exactas del núcleo y del espacio columna. Para la imagen debemos seleccionar columnas de la matriz original, no de su RREF.

In [ ]:
base_nucleo = A.nullspace()
base_imagen = A.columnspace()
rango = A.rank()
nulidad = len(base_nucleo)

print("Base del núcleo:")
for q in base_nucleo:
    sp.pprint(q.T)
print("Base de la imagen:")
for q in base_imagen:
    sp.pprint(q.T)
print("rango =", rango, "; nulidad =", nulidad)

assert all(A * q == sp.zeros(2, 1) for q in base_nucleo)
assert rango + nulidad == A.cols
assert rango == 2

Como el núcleo no es trivial, $T$ no es inyectiva. Como el rango coincide con la dimensión del codominio, $T$ sí es sobreyectiva.

## 3. Una transformación definida sobre una base

Sea $\mathcal B=(b_1,b_2)$ una base de $\mathbb R^2$. Si especificamos $L(b_1)$ y $L(b_2)$, queda definida una única transformación lineal.

In [ ]:
b1 = sp.Matrix([1, 1])
b2 = sp.Matrix([1, -1])
MB = sp.Matrix.hstack(b1, b2)
w1 = sp.Matrix([2, 0, 1])
w2 = sp.Matrix([0, 1, 1])
W = sp.Matrix.hstack(w1, w2)

# Si x = MB*c, entonces L(x) = W*c = W*MB^{-1}*x.
L_can = sp.simplify(W * MB.inv())
print("Matriz canónica de L:")
sp.pprint(L_can)
assert L_can * b1 == w1
assert L_can * b2 == w2

## 4. Matriz relativa a bases distintas

Ahora usamos $T(x,y,z)=(x+2y,y+z)$, la base $\mathcal B$ del dominio y la base $\mathcal C$ del codominio del ejemplo de teoría.

In [ ]:
A0 = sp.Matrix([[1, 2, 0], [0, 1, 1]])
MB3 = sp.Matrix([[1, 1, 0], [0, 1, 1], [0, 0, 1]])
MC2 = sp.Matrix([[1, 1], [1, 0]])
T_CB = sp.simplify(MC2.inv() * A0 * MB3)
print("[T]_{C <- B} =")
sp.pprint(T_CB)
assert T_CB == sp.Matrix([[0, 1, 2], [1, 2, 0]])

In [ ]:
# Construcción directa: columna j = coordenadas en C de T(b_j).
columnas_directas = [MC2.inv() * A0 * MB3.col(j) for j in range(MB3.cols)]
T_directa = sp.Matrix.hstack(*columnas_directas)
assert T_directa == T_CB

x_B = sp.Matrix([2, -1, 3])
x_can = MB3 * x_B
Tx_C = T_CB * x_B
Tx_can = MC2 * Tx_C
print("[x]_B =", list(x_B), "; x =", list(x_can))
print("[T(x)]_C =", list(Tx_C), "; T(x) =", list(Tx_can))
assert Tx_can == A0 * x_can

## 5. Una matriz para la derivación

Con las bases $\mathcal B=(1,t,t^2)$ de $\mathcal P_2$ y $\mathcal C=(1,t)$ de $\mathcal P_1$, cada columna contiene las coordenadas de la derivada de un elemento de $\mathcal B$.

In [ ]:
D = sp.Matrix([[0, 1, 0], [0, 0, 2]])
p_B = sp.Matrix([3, -2, 5])  # p(t)=3-2t+5t^2
Dp_C = D * p_B
print("[p']_C =", list(Dp_C))
print("Base del núcleo de D:", [list(q) for q in D.nullspace()])
print("rango(D) =", D.rank())
assert Dp_C == sp.Matrix([-2, 10])
assert D.rank() + len(D.nullspace()) == 3

## 6. Composición: el orden del producto importa

In [ ]:
# T: R^2 -> R^3, S: R^3 -> R^2
MT = sp.Matrix([[1, 0], [0, 1], [1, 1]])
MS = sp.Matrix([[1, 0, 1], [0, 1, -1]])
M_ST = MS * MT  # S o T: R^2 -> R^2
M_TS = MT * MS  # T o S: R^3 -> R^3
print("Matriz de S o T:")
sp.pprint(M_ST)
print("Matriz de T o S:")
sp.pprint(M_TS)
assert M_ST.shape == (2, 2)
assert M_TS.shape == (3, 3)
assert M_ST != M_TS

## 7. Isomorfismo e inversa

En dimensión finita e igual, basta comprobar rango completo, determinante no nulo o núcleo trivial.

In [ ]:
G = sp.Matrix([[2, 1], [1, 1]])
G_inv = G.inv()
print("det(G) =", G.det())
print("G^{-1} =")
sp.pprint(G_inv)
assert G.det() != 0
assert G * G_inv == sp.eye(2)
assert G_inv * G == sp.eye(2)

## 8. Visualización: una transformación actúa sobre toda la cuadrícula

La figura ayuda a distinguir la transformación geométrica de la matriz que usamos para calcularla.

In [ ]:
M = np.array([[1.0, 0.7], [0.2, 1.3]])
valores = np.linspace(-2, 2, 9)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for c in valores:
    horizontal = np.vstack([np.linspace(-2, 2, 100), np.full(100, c)])
    vertical = np.vstack([np.full(100, c), np.linspace(-2, 2, 100)])
    axes[0].plot(horizontal[0], horizontal[1], color="#8aa9c2", linewidth=0.7)
    axes[0].plot(vertical[0], vertical[1], color="#8aa9c2", linewidth=0.7)
    Mh, Mv = M @ horizontal, M @ vertical
    axes[1].plot(Mh[0], Mh[1], color="#14578c", linewidth=0.7)
    axes[1].plot(Mv[0], Mv[1], color="#be2d37", linewidth=0.7)
for ax, title in zip(axes, ["Cuadrícula original", "Imagen mediante T(x)=Mx"]):
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## 9. Actividades

1. Modifica una columna de la matriz $A$ y predice cómo cambian el núcleo y la imagen antes de calcularlos.
2. Define una transformación $\mathcal P_3\to\mathbb R^2$ mediante $p\mapsto(p(0),p(1))$ y construye su matriz.
3. Cambia las bases $\mathcal B$ y $\mathcal C$ del ejemplo y verifica que la transformación no cambia aunque sí lo haga su matriz.
4. Busca un ejemplo donde $S\circ T$ sea invertible pero $T\circ S$ no pueda serlo por su tamaño o rango.
5. Sustituye la matriz de la visualización por una proyección, una rotación y una matriz singular. Describe cada imagen.
6. Redacta sin código la propiedad teórica verificada por cada `assert`.